# DTW Validation Data Extraction — Akamine et al. Dataset (Body)
**contact: w.pouw@tilburguniversity.edu**

## Steps
1. Extract raw gesture-level body time series → one CSV per gesture event

---
## Step 1 — Extract raw body gesture segments

Trims raw MediaPipe **body** time series to each annotated gesture window.  
**No smoothing or interpolation** — data stored as-is from the sensor.  
`visibility_*` columns are dropped; only `X_`, `Y_`, `Z_` retained.

**Output (`ValidationDataAkamine/processed/`):**
- `timeseries/{comparison_id}_{ts_num}_body.csv` — one file per gesture event (ts_num 1 or 2)
- `gesture_annotation.csv` — shared annotation file (also written by the hands script)

In [ ]:
import numpy as np
import pandas as pd
import os
from tqdm import tqdm

TS_DIRS  = [
    "../osfstorage-archive (1)/output_timeseries_study2_1/",
    "../osfstorage-archive (1)/output_timeseries_study2_2/",
]
ANNO_CSV = "../github-archive/kinematics/study2/data/elan_annotation/gestural_alignment_processed.csv"
BASE_CSV = "../github-archive/kinematics/study2/data/processed/05_dtw_distance/dtw_distance.csv"
OUT_DIR  = "../processed/"
TS_OUT   = OUT_DIR + "timeseries/"
os.makedirs(TS_OUT, exist_ok=True)

pair_dir = {}
for ts_dir in TS_DIRS:
    for f in os.listdir(ts_dir):
        if os.path.isdir(os.path.join(ts_dir, f)):
            pid = int(f.split('_')[0])
            pair_dir[pid] = ts_dir

available_pairs = sorted(pair_dir.keys())
anno = pd.read_csv(ANNO_CSV)
anno = anno[anno['pair'].isin(available_pairs)].reset_index(drop=True)
print(f"Available pairs: {available_pairs}")
print(f"Comparisons to process: {len(anno)}")

In [ ]:
# body keypoint columns: X/Y/Z only, drop visibility_* columns
sample_file = os.path.join(pair_dir[available_pairs[0]],
                           f"{available_pairs[0]:03d}_a",
                           f"{available_pairs[0]:03d}_a_body.csv")
all_cols = pd.read_csv(sample_file, nrows=0).columns.tolist()
KP_COLS  = [c for c in all_cols if c != 'time' and not c.startswith('visibility_')]
print(f"Body keypoint columns: {len(KP_COLS)} ({len(KP_COLS)//3} xyz-triplets)")

In [ ]:
_ts_cache = {}

def load_segment(pair_id: int, speaker: str, begin_ms: float, end_ms: float):
    """Return raw body gesture segment (frames x KP_COLS) or None."""
    prefix = f"{pair_id:03d}_{speaker.lower()}"
    if prefix not in _ts_cache:
        ts_dir = pair_dir.get(pair_id)
        path   = os.path.join(ts_dir, prefix, f"{prefix}_body.csv") if ts_dir else None
        if not path or not os.path.exists(path):
            _ts_cache[prefix] = None
        else:
            try:
                _ts_cache[prefix] = pd.read_csv(path, usecols=['time'] + KP_COLS)
            except Exception:
                _ts_cache[prefix] = pd.read_csv(path, usecols=['time'] + KP_COLS,
                                                 engine='python', on_bad_lines='skip')
    ts = _ts_cache[prefix]
    if ts is None:
        return None
    seg = ts[(ts['time'] >= begin_ms) & (ts['time'] <= end_ms)][KP_COLS].copy()
    return seg.reset_index(drop=True) if len(seg) >= 3 else None

In [ ]:
anno_path  = OUT_DIR + 'gesture_annotation.csv'
n_existing = len([f for f in os.listdir(TS_OUT) if f.endswith('_body.csv')])

if n_existing > 0:
    print(f"Dataset already exists ({n_existing} files) — skipping extraction.")
else:
    baseline  = pd.read_csv(BASE_CSV, usecols=['comparison_id', 'average_distance'])
    ANNO_KEEP = ['comparison_id', 'pair', 'referent',
                 'speaker_1', 'speaker_2', 'round_1', 'round_2',
                 'duration_1_adj', 'duration_2_adj', 'iconic_1', 'iconic_2', 'hands_dtw']

    valid_rows, skipped = [], 0
    for _, row in tqdm(anno.iterrows(), total=len(anno), desc="Extracting"):
        s1 = load_segment(row['pair'], row['speaker_1'],
                          row['begin_time_1_adj'], row['end_time_1_adj'])
        s2 = load_segment(row['pair'], row['speaker_2'],
                          row['begin_time_2_adj'], row['end_time_2_adj'])
        if s1 is None or s2 is None:
            skipped += 1; continue
        cid = row['comparison_id']
        s1.to_csv(f"{TS_OUT}{cid}_1_body.csv", index=False)
        s2.to_csv(f"{TS_OUT}{cid}_2_body.csv", index=False)
        valid_rows.append(row[ANNO_KEEP])

    # write annotation only if not already present (hands script may have written it)
    if not os.path.exists(anno_path):
        df_anno = pd.DataFrame(valid_rows).reset_index(drop=True)
        df_anno = df_anno.merge(baseline, on='comparison_id', how='left')
        df_anno.to_csv(anno_path, index=False)

    n_files = len([f for f in os.listdir(TS_OUT) if f.endswith('_body.csv')])
    seg_mb  = sum(os.path.getsize(TS_OUT+f) for f in os.listdir(TS_OUT)) / 1e6
    print(f"Done. Skipped {skipped} | Saved {len(valid_rows)} comparisons")
    print(f"timeseries/ body files: {n_files}, folder total: {seg_mb:.1f} MB")

In [ ]:
import matplotlib.pyplot as plt

df_anno = pd.read_csv(anno_path)
print(f"{len(df_anno)} comparisons")
print(df_anno[['comparison_id','pair','referent','hands_dtw','average_distance']].head())

cid  = df_anno['comparison_id'].iloc[0]
info = df_anno.iloc[0]
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for ts_num, ax in zip([1, 2], axes):
    seg = pd.read_csv(f"{TS_OUT}{cid}_{ts_num}_body.csv")
    for col in ['X_RIGHT_WRIST', 'Y_RIGHT_WRIST', 'X_LEFT_WRIST', 'Y_LEFT_WRIST',
                'X_RIGHT_ELBOW', 'Y_RIGHT_ELBOW']:
        if col in seg.columns:
            ax.plot(seg[col].values, label=col.replace('_', ' '))
    speaker = info['speaker_1'] if ts_num == 1 else info['speaker_2']
    ax.set_title(f"Gesture {ts_num} — speaker {speaker} (raw body)")
    ax.set_xlabel('frame'); ax.legend(fontsize=7)
plt.tight_layout()
plt.show()